# 04 — Modelo Comparativo: SVM
Comparación Random Forest vs SVM — CRM Sales Opportunities

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, roc_auc_score, RocCurveDisplay
)
from paths import PROCESSED_DIR

X_train = pd.read_csv(PROCESSED_DIR / "X_train_scaled.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test_scaled.csv")
X_test_rf = pd.read_csv(PROCESSED_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()
rf_model = joblib.load(PROCESSED_DIR / "rf_model.joblib")

In [ ]:
svm_pipeline = Pipeline([
    ("svm", SVC(
        kernel="rbf", C=1.0, gamma="scale",
        class_weight="balanced", probability=True, random_state=42
    ))
])

svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)
y_proba_svm = svm_pipeline.predict_proba(X_test)[:, 1]

print("=== SVM (RBF) ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"F1 macro : {f1_score(y_test, y_pred_svm, average='macro'):.4f}")
print(f"AUC-ROC  : {roc_auc_score(y_test, y_proba_svm):.4f}")
print(classification_report(y_test, y_pred_svm, target_names=["Lost", "Won"]))

In [ ]:
y_pred_rf = rf_model.predict(X_test_rf)
y_proba_rf = rf_model.predict_proba(X_test_rf)[:, 1]

print("=== Random Forest ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"F1 macro : {f1_score(y_test, y_pred_rf, average='macro'):.4f}")
print(f"AUC-ROC  : {roc_auc_score(y_test, y_proba_rf):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in [
    (axes[0], y_pred_rf, "Random Forest"),
    (axes[1], y_pred_svm, "SVM (RBF)")
]:
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Lost", "Won"], yticklabels=["Lost", "Won"])
    ax.set_title(f"Matriz de confusión — {title}")
    ax.set_ylabel("Real"); ax.set_xlabel("Predicción")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_predictions(y_test, y_proba_rf, ax=ax, name="Random Forest")
RocCurveDisplay.from_predictions(y_test, y_proba_svm, ax=ax, name="SVM (RBF)")
ax.plot([0, 1], [0, 1], "k--", label="Aleatorio (AUC=0.50)")
ax.set_title("Comparación ROC — CRM Won/Lost")
ax.legend()
plt.tight_layout()
plt.show()

joblib.dump(svm_pipeline, PROCESSED_DIR / "svm_model.joblib")
print("Modelo SVM guardado.")